# Danish Mushroom Classifier 🍄
Train an image classifier for **six common Danish mushrooms** from ~360 citizen-science photos taken *in Denmark*, using **transfer learning**: borrow a network pre-trained on 1.2M ImageNet photos and teach only its last layer. Then swap in **your own photos** (birds, your cat vs. your neighbour's cat, Aalborg buildings…).

> [!warning] Never eat a mushroom because a model said so
> This is a machine-learning exercise, not a field guide. You'll see below that the model confuses the **deadly death cap** with edible species. In Denmark, get mushrooms checked by a *svampekontrollant* (mushroom inspector).

Guide note: [[Danish Mushroom Classifier]] · background: [[CNN]], [[CNN Architectures]], [[Computer Vision Overview]]

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import json, pathlib, time, urllib.request, urllib.parse, concurrent.futures as cf, numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F, torchvision
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
torch.manual_seed(0); torch.set_num_threads(4)
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing

# iNaturalist taxon ids (look up others at inaturalist.org – the number in the taxon URL)
SPECIES = {"Rød fluesvamp (fly agaric)": 48715, "Karl Johan (porcini)": 48701, "Kantarel (chanterelle)": 47347,
           "Grøn fluesvamp (DEATH CAP)": 52135, "Svovlgul svovlhat (sulphur tuft)": 48767, "Stinksvamp (stinkhorn)": 54594}
PER_CLASS, DENMARK = 60, 8051                          # iNaturalist place id for Denmark
MY_PHOTOS = pathlib.Path("data/my_photos")             # your own: data/my_photos/<class name>/*.jpg  → used instead if it exists
ROOT = pathlib.Path("data/mushrooms")

def download_inaturalist():
    rows = []
    for name, taxon in SPECIES.items():
        q = dict(taxon_id=taxon, place_id=DENMARK, quality_grade="research", photos="true",
                 photo_license="cc0,cc-by,cc-by-nc", per_page=PER_CLASS, order_by="votes")
        req = urllib.request.Request("https://api.inaturalist.org/v1/observations?" + urllib.parse.urlencode(q), headers={"User-Agent": "UniNotes study notebook"})
        for o in json.load(urllib.request.urlopen(req))["results"]:
            p = o["photos"][0]
            rows.append(dict(cls=name, file=str(ROOT / name / f"{o['id']}.jpg"), url=p["url"].replace("square", "small"),
                             attribution=p["attribution"], license=p["license_code"], observation=f"https://www.inaturalist.org/observations/{o['id']}"))
        time.sleep(1)                                    # be polite to the API
    def get(r):
        f = pathlib.Path(r["file"]); f.parent.mkdir(parents=True, exist_ok=True)
        if not f.exists(): f.write_bytes(urllib.request.urlopen(r["url"]).read())
    with cf.ThreadPoolExecutor(8) as ex: list(ex.map(get, rows))
    pd.DataFrame(rows).to_csv(ROOT / "credits.csv", index=False)

if MY_PHOTOS.exists() and any(MY_PHOTOS.iterdir()):
    root = MY_PHOTOS; print("Using YOUR photos from", root)
else:
    if not (ROOT / "credits.csv").exists(): download_inaturalist()
    root = ROOT
classes = sorted(d.name for d in root.iterdir() if d.is_dir())
paths, labels = [], []
for c, name in enumerate(classes):
    for f in sorted((root / name).glob("*")):
        if f.suffix.lower() in (".jpg", ".jpeg", ".png"): paths.append(str(f)); labels.append(c)
labels = np.array(labels)
p_train, p_test, y_train, y_test = train_test_split(paths, labels, test_size=0.3, stratify=labels, random_state=0)
print(f"{len(paths)} photos, {len(classes)} classes, {len(p_train)} train / {len(p_test)} test, chance = {1/len(classes):.0%}")
fig, ax = plt.subplots(1, len(classes), figsize=(2.3 * len(classes), 2.6))
for a, c in zip(ax, range(len(classes))):
    a.imshow(Image.open(paths[list(labels).index(c)])); a.set_title(classes[c].split(" (")[0], fontsize=9); a.axis("off")
plt.show()

## 0. Baseline: a small CNN from scratch
252 photos is *nothing* for a CNN that starts from random weights. Let's see (30 epochs, 64×64 pixels, horizontal flips, ~10 s).

In [ ]:
small_tf = torchvision.transforms.Compose([torchvision.transforms.Resize((64, 64)), torchvision.transforms.ToTensor()])
Xs_tr = torch.stack([small_tf(Image.open(p).convert("RGB")) for p in p_train]); Xs_te = torch.stack([small_tf(Image.open(p).convert("RGB")) for p in p_test])
scratch = nn.Sequential(nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                        nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(128, len(classes)))
opt = torch.optim.Adam(scratch.parameters(), 1e-3); yt = torch.tensor(y_train)
for epoch in range(30):
    for idx in torch.randperm(len(Xs_tr)).split(32):
        xb = torch.where(torch.rand(len(idx), 1, 1, 1) < 0.5, Xs_tr[idx].flip(3), Xs_tr[idx])
        loss = F.cross_entropy(scratch(xb), yt[idx]); opt.zero_grad(); loss.backward(); opt.step()
scratch.eval(); scratch_acc = float((scratch(Xs_te).argmax(1).numpy() == y_test).mean())
print(f"from-scratch CNN test accuracy: {scratch_acc:.1%}  (chance {1/len(classes):.0%})")

## 1. Borrow a backbone: features from ResNet-18
A ResNet-18 trained on ImageNet already detects edges, textures, gills, caps… Replace its final classification layer with `nn.Identity()` and it becomes a **feature extractor**: every photo → a 512-number vector. Use the weights' own preprocessing (`weights.transforms()`: resize, centre-crop 224, normalise).

Implement `extract_features`: load each image as RGB, apply `transform`, stack in batches, run `backbone` under `torch.no_grad()`, and return an `(N, 512)` NumPy array.

In [ ]:
weights = torchvision.models.ResNet18_Weights.IMAGENET1K_V1
backbone = torchvision.models.resnet18(weights=weights); backbone.fc = nn.Identity(); backbone.eval()
transform = weights.transforms()

def extract_features(paths, backbone, transform, batch_size=32):
    # TODO 1: batches of transformed images → backbone → concatenate → numpy (N, 512)
    raise NotImplementedError  # TODO 1

In [ ]:
def _t_feat():
    f = extract_features(p_train[:5], backbone, transform, batch_size=2)
    ref = backbone(transform(Image.open(p_train[3]).convert("RGB"))[None]).detach().numpy()[0]
    return f.shape == (5, 512) and np.allclose(f[3], ref, atol=1e-4)
check("features", _t_feat, "Expect shape (5, 512) and row 3 equal to the backbone output for image 3 (same order, batches concatenated).")
if ready("features"):
    t = time.time(); F_train, F_test = extract_features(p_train, backbone, transform), extract_features(p_test, backbone, transform)
    print(f"extracted {len(F_train) + len(F_test)} feature vectors in {time.time()-t:.0f}s")

## 2. Linear probe
Train **logistic regression** on the frozen features: that's transfer learning in its simplest form. Return the fitted classifier and its test accuracy.

In [ ]:
def linear_probe(X_train, y_train, X_test, y_test, C=1.0):
    # TODO 2: LogisticRegression(max_iter=3000, C=C) → fit → (clf, test accuracy)
    raise NotImplementedError  # TODO 2

In [ ]:
check("linear probe", lambda: linear_probe(np.array([[0.0], [0.1], [1.0], [1.1]]), np.array([0, 0, 1, 1]), np.array([[0.05], [1.05]]), np.array([0, 1]))[1] == 1.0,
      "On a trivially separable toy set the accuracy must be 1.0; return (clf, accuracy).")
if ready("features", "linear probe"):
    clf, probe_acc = linear_probe(F_train, y_train, F_test, y_test)
    print(f"linear probe on ResNet-18 features: {probe_acc:.1%}   vs from scratch: {scratch_acc:.1%}")
    if root == ROOT:
        check("transfer learning wins", lambda: probe_acc >= 0.75 and probe_acc > scratch_acc + 0.3,
              "The linear probe should reach ≥ 75 % and beat the from-scratch CNN by 30+ points.")

## 3. Which mistakes, and how bad?
Accuracy treats every error the same. For mushrooms, *death cap → edible* is the one that kills. Implement `top_confusions(cm, classes, n)`: the `n` largest **off-diagonal** cells of the confusion matrix as `(count, true class, predicted class)`, largest first.

In [ ]:
def top_confusions(cm, classes, n=5):
    # TODO 3: off-diagonal (count, true, predicted) triples with count > 0, sorted by count descending, first n
    raise NotImplementedError  # TODO 3

In [ ]:
check("confusions", lambda: top_confusions(np.array([[5, 2, 0], [0, 4, 3], [1, 0, 6]]), ["a", "b", "c"], 2) == [(3, "b", "c"), (2, "a", "b")],
      "For [[5,2,0],[0,4,3],[1,0,6]] the two biggest mistakes are b→c (3) and a→b (2).")
if ready("features", "linear probe", "confusions"):
    pred = clf.predict(F_test); proba = clf.predict_proba(F_test); cm = confusion_matrix(y_test, pred)
    short = [c.split(" (")[0] for c in classes]
    plt.figure(figsize=(5.5, 4.5)); plt.imshow(cm, cmap="Blues"); plt.xticks(range(len(short)), short, rotation=45, ha="right"); plt.yticks(range(len(short)), short)
    for i in range(len(short)):
        for j in range(len(short)): plt.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
    plt.xlabel("predicted"); plt.ylabel("true"); plt.title("confusion matrix (test)"); plt.show()
    print("per-class recall:", {s: f"{cm[i, i] / cm[i].sum():.0%}" for i, s in enumerate(short)})
    for cnt, t, p in top_confusions(cm, classes): print(f"  {cnt}×  {t}  →  predicted {p}")
    wrong = np.where(pred != y_test)[0]; wrong = wrong[np.argsort(-proba[wrong].max(1))][:6]
    fig, ax = plt.subplots(1, len(wrong), figsize=(2.5 * len(wrong), 2.9))
    for a, i in zip(np.atleast_1d(ax), wrong):
        a.imshow(Image.open(p_test[i])); a.axis("off")
        a.set_title(f"true: {short[y_test[i]]}\npred: {short[pred[i]]} ({proba[i].max():.0%})", fontsize=8)
    plt.suptitle("most confident mistakes"); plt.show()

## 4. Fine-tuning (≈ 15–30 s on a CPU)
Instead of freezing everything, also train the **last residual stage** (`layer4`) with a small learning rate and data augmentation. With so few photos it may or may not beat the linear probe. Measure it, don't assume.

In [ ]:
FINE_TUNE, EPOCHS = True, 3
if FINE_TUNE and ready("features", "linear probe"):
    aug = torchvision.transforms.Compose([torchvision.transforms.RandomResizedCrop(224, scale=(0.6, 1.0)), torchvision.transforms.RandomHorizontalFlip(),
                                          torchvision.transforms.ColorJitter(0.2, 0.2, 0.2), torchvision.transforms.ToTensor(),
                                          torchvision.transforms.Normalize(weights.transforms().mean, weights.transforms().std)])
    torch.manual_seed(0)
    ft = torchvision.models.resnet18(weights=weights); ft.fc = nn.Linear(512, len(classes))
    for name, prm in ft.named_parameters(): prm.requires_grad = name.startswith(("layer4", "fc"))
    opt = torch.optim.AdamW([p for p in ft.parameters() if p.requires_grad], lr=3e-4, weight_decay=1e-4)
    imgs = [Image.open(p).convert("RGB") for p in p_train]; t = time.time()
    for epoch in range(EPOCHS):
        ft.train()
        for idx in torch.randperm(len(imgs)).split(32):
            xb = torch.stack([aug(imgs[i]) for i in idx]); loss = F.cross_entropy(ft(xb), yt[idx])
            opt.zero_grad(); loss.backward(); opt.step()
        print(f"epoch {epoch + 1}: last batch loss {loss.item():.3f} ({time.time()-t:.0f}s)")
    ft.eval()
    with torch.no_grad():
        logits = torch.cat([ft(torch.stack([transform(Image.open(p).convert("RGB")) for p in p_test[s:s + 32]])) for s in range(0, len(p_test), 32)])
    ft_acc = float((logits.argmax(1).numpy() == y_test).mean())
    print(f"fine-tuned layer4: {ft_acc:.1%}   linear probe: {probe_acc:.1%}   from scratch: {scratch_acc:.1%}")

In [ ]:
if root == ROOT and (ROOT / "credits.csv").exists():
    credits = pd.read_csv(ROOT / "credits.csv")
    print(f"Photos: iNaturalist observers, licences {sorted(credits.license.str.upper().unique())}. Full attribution per photo in {ROOT/'credits.csv'}.")
    print(credits[["cls", "attribution", "observation"]].head(3).to_string(index=False))

<details><summary>▶ Solution</summary>

```python
def extract_features(paths, backbone, transform, batch_size=32):
    feats = []
    with torch.no_grad():
        for start in range(0, len(paths), batch_size):
            batch = torch.stack([transform(Image.open(p).convert("RGB")) for p in paths[start:start + batch_size]])
            feats.append(backbone(batch))
    return torch.cat(feats).numpy()
```

```python
def linear_probe(X_train, y_train, X_test, y_test, C=1.0):
    clf = LogisticRegression(max_iter=3000, C=C).fit(X_train, y_train)
    return clf, float((clf.predict(X_test) == y_test).mean())
```

```python
def top_confusions(cm, classes, n=5):
    pairs = [(int(cm[i, j]), classes[i], classes[j]) for i in range(len(classes)) for j in range(len(classes)) if i != j and cm[i, j] > 0]
    return sorted(pairs, key=lambda p: -p[0])[:n]
```
</details>

## Make it yours (stretch goals)
- **Your own photos:** put 20–50 photos per class in `data/my_photos/<class>/` and rerun. Ideas: Danish birds at your feeder, your bike vs. other bikes, AAU buildings, handwritten Danish letters æ/ø/å.
- **Danish birds instead:** change `SPECIES`, e.g. solsort (*Turdus merula*), musvit (*Parus major*), gråspurv (*Passer domesticus*), skarv (*Phalacrocorax carbo*). Look up the taxon ids on inaturalist.org.
- **Cost-sensitive decisions:** refuse to answer ("ask an expert") whenever `max(proba) < threshold` or the top-2 includes the death cap. Plot coverage vs. accuracy.
- **Grad-CAM:** which pixels made the model say *death cap*? Is it the mushroom or the forest floor? (Shortcut learning, see [[Explainability and Fairness]].)
- **Bigger backbone:** `convnext_tiny` or a CLIP image encoder as the feature extractor. Is the linear probe better?